# Garcia — laboratório VoxCPM2 (Kaggle)

Notebook experimental para síntese e clonagem autorizada de voz em GPU. Os pesos não são enviados ao APK nem executados no Android.

**Privacidade:** use somente áudio da sua própria voz ou de pessoa que autorizou expressamente a clonagem. Não use para se passar por terceiros, fraude ou desinformação.

## 1. Verificar o ambiente
No Kaggle, selecione **Settings → Accelerator → GPU** antes de executar. A disponibilidade e a memória da GPU dependem da quota/conta; este notebook não contorna limites do provedor.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except Exception as exc:
    print('Torch check failed:', type(exc).__name__, str(exc)[:300])
if not ('/kaggle' in os.getcwd() or os.path.exists('/kaggle')):
    print('Aviso: ambiente fora do Kaggle; prossiga somente se configurou GPU CUDA compatível.')

## 2. Instalar o VoxCPM
A instalação pode levar alguns minutos e baixar dependências grandes. Se o Kaggle reiniciar a sessão, execute novamente as células em ordem.

In [ ]:
%pip install -q voxcpm soundfile librosa


## 3. Configurar áudio de referência e texto
Adicione um Dataset privado ao notebook contendo um WAV/MP3 autorizado. Informe abaixo o caminho exato em `/kaggle/input/...`. Prefira gravação limpa, sem música ou outras vozes.

In [ ]:
from pathlib import Path
REFERENCE_AUDIO = '/kaggle/input/SEU-DATASET/sua-voz.wav'  # editar
OUTPUT_AUDIO = '/kaggle/working/garcia_voxcpm2_clone.wav'
TEXT_TO_SPEAK = 'Olá. Este é um teste de clonagem de voz da Garcia em português brasileiro.'
REFERENCE_TRANSCRIPT = ''  # opcional: transcrição exata do áudio de referência para clonagem de maior fidelidade
assert Path(REFERENCE_AUDIO).is_file(), f'Arquivo de referência não encontrado: {REFERENCE_AUDIO}'
assert TEXT_TO_SPEAK.strip(), 'Informe o texto que será sintetizado.'
print('Referência encontrada:', Path(REFERENCE_AUDIO).name)
print('Tamanho (MiB):', round(Path(REFERENCE_AUDIO).stat().st_size / 1024**2, 2))

## 4. Preparar referência em mono 16 kHz
O VoxCPM2 documenta áudio de referência em 16 kHz e saída em 48 kHz. A conversão abaixo preserva o arquivo original e cria uma cópia temporária.

In [ ]:
import librosa, soundfile as sf
reference_audio, _ = librosa.load(REFERENCE_AUDIO, sr=16000, mono=True)
REFERENCE_16K = '/kaggle/working/reference_16k_mono.wav'
sf.write(REFERENCE_16K, reference_audio, 16000, subtype='PCM_16')
print('Referência preparada:', REFERENCE_16K, '| duração:', round(len(reference_audio)/16000, 2), 's')
if len(reference_audio) < 2 * 16000:
    print('Aviso: referência muito curta; grave uma amostra mais longa e limpa para melhores resultados.')

## 5. Carregar modelo e gerar voz clonada
A primeira execução baixa os pesos do repositório oficial `openbmb/VoxCPM2`; o tamanho do download e o tempo de carga dependem da rede e da GPU. Não armazene tokens no notebook.

In [ ]:
import torch, soundfile as sf
from voxcpm import VoxCPM

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA não detectada. Ative uma GPU no Kaggle e reinicie a sessão.')

model = VoxCPM.from_pretrained('openbmb/VoxCPM2', load_denoiser=False)

if REFERENCE_TRANSCRIPT.strip():
    audio = model.generate(
        text=TEXT_TO_SPEAK,
        prompt_wav_path=REFERENCE_16K,
        prompt_text=REFERENCE_TRANSCRIPT.strip(),
        reference_wav_path=REFERENCE_16K,
        cfg_value=2.0,
        inference_timesteps=10,
    )
else:
    audio = model.generate(
        text=TEXT_TO_SPEAK,
        reference_wav_path=REFERENCE_16K,
        cfg_value=2.0,
        inference_timesteps=10,
    )

sample_rate = model.tts_model.sample_rate
sf.write(OUTPUT_AUDIO, audio, sample_rate)
print('Áudio gerado:', OUTPUT_AUDIO)
print('Sample rate:', sample_rate, '| duração:', round(len(audio)/sample_rate, 2), 's')


## 6. Ouvir e validar resultado
Compare a pronúncia, timbre, ritmo e artefatos com a referência. Resultado de laboratório não significa integração pronta na Garcia.

In [ ]:
from IPython.display import Audio, display
display(Audio(filename=OUTPUT_AUDIO))


## 7. Exportar artefato
O WAV fica em `/kaggle/working/garcia_voxcpm2_clone.wav` e pode ser baixado pela área Output do Kaggle. A integração automática com a Garcia/servidor ainda não está habilitada nesta etapa; não publique áudio privado em Dataset público.

In [ ]:
from pathlib import Path
p = Path(OUTPUT_AUDIO)
assert p.is_file() and p.stat().st_size > 44, 'Arquivo WAV não foi criado corretamente.'
print('Pronto para download:', p.name, '| MiB:', round(p.stat().st_size/1024**2, 2))
